In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import mne
import os
import gc
import numpy as np
from scipy.signal import hilbert
import matplotlib.pyplot as plt
import traceback

\participants = ['3', '6', '7', '8', '9', '10', '11', '12', '13', '14',
                '15', '16', '17', '18', '19', '20', '21', '22', '23',
                '24', '25', '26', '27']


In [ ]:
# Set your EEG data root directory

root_dir = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

def preprocess_and_save_with_ica(participant):
    participant_path = os.path.join(root_dir, participant)
    print(f"\n>>> Processing participant folder: {participant_path}")

    if not os.path.exists(participant_path):
        print(f"[ERROR] Folder does not exist: {participant_path}")
        return

    set_files = [f for f in os.listdir(participant_path) if f.endswith('.set')]
    print(f"Found {len(set_files)} .set files: {set_files}")

    if not set_files:
        print(f"[ERROR] No .set files found in: {participant_path}")
        return

    for set_file in set_files:
        set_name = os.path.splitext(set_file)[0]
        raw_path = os.path.join(participant_path, set_file)
        fif_path = os.path.join(participant_path, f'{set_name}_clean_raw.fif')
        ica_path = os.path.join(participant_path, f'{set_name}-ica.fif')

        print(f"\n--- Processing file: {set_file} ---")

        if os.path.exists(fif_path) and os.path.exists(ica_path):
            print("Already preprocessed, skipping.")
            continue

        try:
            raw = mne.io.read_raw_eeglab(raw_path, preload=True)
            montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
            raw.set_montage(montage)

            # Re-reference to LM/RM (E57, E100)
            try:
                lm = raw.copy().pick(['E57']).get_data()
                rm = raw.copy().pick(['E100']).get_data()
                mastoid_avg = (lm + rm) / 2
                raw._data = raw.get_data() - mastoid_avg
                print("Re-referenced to LM/RM.")
            except Exception as e:
                print(f"Referencing error: {e}")
                traceback.print_exc()
                continue

            # Detect bad channels (amplitude-based)
            picks_all = mne.pick_types(raw.info, eeg=True, exclude=[])
            data = raw.get_data(picks_all)
            ch_stds = data.std(axis=1)
            threshold = 5 * ch_stds.mean()
            bad_channels = [raw.ch_names[p] for p, val in enumerate(ch_stds) if val > threshold]
            raw.info['bads'] = bad_channels
            print(f"Bad channels detected: {bad_channels}")
            # plot .raw files and check manually and save all bad channels as a list


            raw.interpolate_bads(reset_bads=True)
            print("Interpolated bad channels.")

            raw.notch_filter(freqs=60)
            print("Applied 60 Hz notch filter.")

            # ICA preparation
            raw_for_ica = raw.copy().filter(0.1, None)
            raw_for_ica.crop(tmin=0, tmax=600)  # Optional: first 10 minutes only
            picks_ica = mne.pick_types(raw_for_ica.info, eeg=True, exclude='bads')

            # ICA with exactly 20 components
            ica = mne.preprocessing.ICA(n_components=20, method='fastica', random_state=97, max_iter=500)
            ica.fit(raw_for_ica, picks=picks_ica)
            print("ICA fitted with 20 components.")

            # Save ICA solution
            ica.save(ica_path, overwrite=True)
            print(f"Saved ICA: {ica_path}")

            # Save ICA component scalp map
            fig_comp = ica.plot_components(inst=raw, show=False)
            fig_comp_path = os.path.join(participant_path, f"{set_name}_ica_components.png")
            fig_comp.savefig(fig_comp_path, dpi=150)
            plt.close(fig_comp)
            print(f"Saved ICA component scalp map: {fig_comp_path}")

            # Save ICA time series (component activations)
            fig_sources = ica.plot_sources(raw, show=False)
            fig_src_path = os.path.join(participant_path, f"{set_name}_ica_sources.png")
            fig_sources.savefig(fig_src_path, dpi=150)
            plt.close(fig_sources)
            print(f"Saved ICA time series plot: {fig_src_path}")

            # Save cleaned raw data
            raw.save(fif_path, overwrite=True)
            print(f"Saved clean raw data: {fif_path}")

            # Cleanup
            del raw, raw_for_ica, ica, fig_comp, fig_sources
            gc.collect()

        except Exception as e:
            print(f"ERROR processing {set_file}: {e}")
            traceback.print_exc()
            continue

def batch_process_all_participants():
    participants = [p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))]
    print(f"\n>>> Found {len(participants)} participant folders: {participants}")
    for participant in participants:
        print(f"\n=== Starting: {participant} ===")
        preprocess_and_save_with_ica(participant)


In [ ]:
preprocess_and_save_with_ica('3')
preprocess_and_save_with_ica('6')
preprocess_and_save_with_ica('7')
preprocess_and_save_with_ica('8')
preprocess_and_save_with_ica('10')
preprocess_and_save_with_ica('11')
preprocess_and_save_with_ica('12')
preprocess_and_save_with_ica('13')
preprocess_and_save_with_ica('14')
preprocess_and_save_with_ica('15')
preprocess_and_save_with_ica('17')
preprocess_and_save_with_ica('18')
preprocess_and_save_with_ica('19')
preprocess_and_save_with_ica('20')
preprocess_and_save_with_ica('21')
preprocess_and_save_with_ica('22')
preprocess_and_save_with_ica('24')
preprocess_and_save_with_ica('25')
preprocess_and_save_with_ica('26')
preprocess_and_save_with_ica('27')

In [ ]:
# Root EEG data folder
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
root_dir = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

# ICA components to remove per participant
ica_removal_dict = {
    '3': [8, 14, 16],
    '6': [14, 15, 2,19],
    '7': [6, 7, 14, 18, 19],
    '8': [3, 4, 15, 19],
    '10': [13,16],
    '11': [3, 4, 15, 17],
    '12': [1, 4, 8, 11],
    '13': [0, 4, 7, 15],
    '14': [7],
    '15': [2, 6],
    '17': [1, 2, 6, 7],
    '18': [4],
    '19': [0, 2, 14],
    '20': [7],
    '21': [5],
    '22': [2, 7, 11],
    '24': [0,1,4],
    '25': [7, 8, 9, 10, 11, 12, 13, 14, 18],
    '26': [0, 1, 2, 6, 8, 14],
    '27': [0, 1, 2, 6, 7, 8, 14]
}

def remove_ica_components(participant):
    participant_path = os.path.join(root_dir, participant)

    # Loop through matching files
    fif_files = [f for f in os.listdir(participant_path) if f.endswith('_clean_raw.fif')]

    for fif_file in fif_files:
        try:
            set_name = fif_file.replace('_clean_raw.fif', '')
            fif_path = os.path.join(participant_path, fif_file)
            ica_path = os.path.join(participant_path, f'{set_name}-ica.fif')
            output_path = os.path.join(participant_path, f'{set_name}_ica_removed_raw.fif')

            print(f"\n>>> Processing: {fif_file}")
            print(f"Loading raw from: {fif_path}")
            print(f"Loading ICA from: {ica_path}")

            # Load raw and ICA
            raw = mne.io.read_raw_fif(fif_path, preload=True)
            ica = mne.preprocessing.read_ica(ica_path)

            # Get components to remove
            components = ica_removal_dict.get(participant, [])
            if not components:
                print(f"No ICA components listed for participant {participant}. Skipping.")
                continue

            # Exclude and apply
            ica.exclude = components
            print(f"Excluding components: {components}")
            raw_clean = ica.apply(raw.copy())
            print("ICA components removed.")

            # Save output
            raw_clean.save(output_path, overwrite=True)
            print(f"Saved cleaned file with ICA removed: {output_path}")

        except Exception as e:
            print(f"[ERROR] Failed to process {fif_file}: {e}")
            traceback.print_exc()


In [ ]:
#remove ICA:  8 , 14, 16
remove_ica_components('3')

In [ ]:
#remove ICA:  14,15,2,19
remove_ica_components('6')

In [ ]:
# [3, 4, 15, 19]
remove_ica_components('8')

In [ ]:
# [13,16]
remove_ica_components('10')

In [ ]:
# '11': [3, 4, 15, 17]
remove_ica_components('11')

In [ ]:
# '12': [1, 4, 8, 11]
remove_ica_components('12')

In [ ]:
#remove ICA:  2, 7, 11
remove_ica_components('22')
#remove ICA:  5
remove_ica_components('21')
#remove ICA:  7
remove_ica_components('20')
#remove ICA:  0, 12, 14
remove_ica_components('19')
#remove ICA: 4
remove_ica_components('18')
#remove ICA: 1, 2, 6, 7
remove_ica_components('17')
#remove ICA: 2, 6
remove_ica_components('15')
#remove ICA: 7
remove_ica_components('14')
#remove ICA: 0, 4, 7, 15
remove_ica_components('13')

In [ ]:
#remove ICA: 0, 1, 4
remove_ica_components('24')

In [ ]:
#remove ICA: [7, 8, 9, 10, 11, 12, 13, 14, 18]
remove_ica_components('25')

In [ ]:
#remove ICA: [0, 1, 2, 6, 8, 14]
remove_ica_components('26')

In [ ]:
#remove ICA: [0, 1, 2, 6, 7, 8, 14]
remove_ica_components('27')